In [1]:
import torch
import torch.nn as nn
import os

# 1. 必须重新定义模型结构 (与训练时完全一致)
class TinyCNN(nn.Module):
    def __init__(self, num_classes):
        super(TinyCNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1), nn.BatchNorm2d(16), nn.ReLU(), nn.MaxPool2d(2),
            nn.Dropout2d(0.1),
            nn.Conv2d(16, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Dropout2d(0.1),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 8 * 8, 128),  # 64 channels * 8 * 8 size
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        return self.classifier(self.features(x))

# 2. 配置路径和参数
RESULTS_DIR = "./training_results_tensor"  # 你的结果目录
MODEL_PATH = os.path.join(RESULTS_DIR, "best_model.pth")
ONNX_PATH = os.path.join(RESULTS_DIR, "best_model2.onnx")
NUM_CLASSES = 12  # 你的类别数量
IMAGE_SIZE = 64
DEVICE = torch.device("cpu") # 导出时用 CPU 即可，不需要 GPU

def convert():
    if not os.path.exists(MODEL_PATH):
        print(f"Model file not found: {MODEL_PATH}")
        return

    # 3. 初始化模型并加载权重
    print("Loading model...")
    model = TinyCNN(NUM_CLASSES).to(DEVICE)
    
    # 加载权重 (map_location 确保在 CPU 上也能加载 GPU 训练的权重)
    state_dict = torch.load(MODEL_PATH, map_location=DEVICE)
    model.load_state_dict(state_dict)
    model.eval() # 必须切换到 eval 模式，这会固定 BatchNorm 和 Dropout

    # 4. 创建虚拟输入
    dummy_input = torch.randn(1, 3, IMAGE_SIZE, IMAGE_SIZE).to(DEVICE)

    # 5. 导出
    print(f"Exporting to {ONNX_PATH}...")
    torch.onnx.export(
        model,
        dummy_input,
        ONNX_PATH,
        export_params=True,
        opset_version=11,
        do_constant_folding=True,
        input_names=['input'],
        output_names=['output'],
        dynamic_axes={'input': {0: 'batch_size'}, 'output': {0: 'batch_size'}}
    )
    print("Done!")

if __name__ == "__main__":
    convert()

d:\anaconda3\envs\pytorch1_12_1\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading model...
Exporting to ./training_results_tensor\best_model2.onnx...
Done!
